# 🗣️ Live Transcription with Speaker Labels

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/real_time_streaming/Live_Transcription_Speaker_Labels.ipynb)

Transcribe a live stream as it happens and label **who** is speaking. Works on any RTStream source: an RTSP camera, a push stream, or a YouTube live broadcast.

You will:
1. Connect a live stream with audio.
2. Start transcription with speaker labels.
3. Read the transcript live, line by line.
4. See how speaker labels look in the data, word by word.
5. Switch to the more accurate `pro` model.

**Requirements**
- `videodb` SDK **0.5.4** or newer.
- Any plan. The sample stream here is an RTSP source. A YouTube source needs the Pro plan.

**Options and prices** (per hour of transcription)

| Option | What it does | Price |
|---|---|---|
| `model_name="standard"` (default) | Fast multilingual model | $0.60 |
| `model_name="pro"` | More accurate, better on names and figures | $0.814 |
| `speaker_labels=True` | Labels each record and word with a speaker (`A`, `B`, ...) | +$0.171 |
| `language_code="es"` | Steers the model to one language. Omit it to auto-detect | — |
| `max_speakers=5` | Caps the number of distinct speakers (1–10) | — |

Speaker labels work with `pro`, and with `standard` for English, Spanish, French, German, Portuguese and Italian, or when no language is set.

---
## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q "videodb>=0.5.4"

---
## 🔑 Step 2: Connect to VideoDB

Enter your `VIDEO_DB_API_KEY` when asked. The input is masked.

In [ ]:
import os
from getpass import getpass

import videodb

if not os.environ.get("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Please enter your VideoDB API Key: ")

conn = videodb.connect()
coll = conn.get_collection()
print("Connected to VideoDB")

---
## 📡 Step 3: Connect a Live Stream

We use a public sample: an English interview with several speakers, as an audio-only RTSP stream. To use your own source, change the URL. For a YouTube live broadcast, see [Ingest a YouTube Live Stream](./YouTube_Live_Ingestion.ipynb).

In [ ]:
STREAM_URL = "rtsp://samples.rts.videodb.io:8554/store-cam-audio"
STREAM_NAME = "Interview sample"

stream = coll.connect_rtstream(
    name=STREAM_NAME,
    url=STREAM_URL,
    media_types=["audio"],
)
print(stream.id, stream.status)

#### Already connected the stream? Reconnect with its id instead.

In [ ]:
# stream = coll.get_rtstream("rts-...")

---
## 🎙️ Step 4: Start Transcription with Speaker Labels

The standard model with speaker labels. `max_speakers` should be the number of people you expect: extra voices reuse existing labels.

In [ ]:
stream.start_transcript(language_code="en", speaker_labels=True, max_speakers=4)

status = conn.get(path=f"rtstream/{stream.id}/transcription/status")
print({k: status.get(k) for k in ("status", "model", "speaker_labels", "language")})

---
## 📝 Step 5: Read the Transcript Live

The first lines appear once the stream's worker is connected, usually within a minute or two. A line arrives when a speaker pauses.

In [ ]:
import time
from datetime import datetime


def show_live_transcript(stream, minutes, since=None):
    seen = set()
    deadline = time.time() + minutes * 60
    while time.time() < deadline:
        page = stream.get_transcript(page_size=200, since=since) or {}
        for record in page.get("transcription_records") or []:
            key = (record.get("start"), record.get("text"))
            if key in seen:
                continue
            seen.add(key)
            since = max(since or 0, record.get("start") or 0)
            clock = datetime.fromtimestamp(record["start"]).strftime("%H:%M:%S") if record.get("start") else "--:--:--"
            print(f"[{clock}] Speaker {record.get('speaker') or '?'}: {record.get('text', '')}")
        time.sleep(5)
    print(f"\n{len(seen)} transcript lines")


show_live_transcript(stream, minutes=3)

---
## 🧩 Step 6: See the Speaker Data

A transcript record ends at a pause, not when the speaker changes, so one record can hold two speakers. Its `speaker` is the main one. Every word in `word_timestamps` carries its own `speaker`.

The cell below shows one raw record, then splits every record wherever the word speaker changes, and totals talk time per speaker.

In [ ]:
import json
from collections import defaultdict
from datetime import datetime


def all_records(stream):
    records, page = [], 1
    while True:
        data = stream.get_transcript(page=page, page_size=200) or {}
        records += data.get("transcription_records") or []
        if not data.get("next_page"):
            return records
        page += 1


def speaker_segments(record):
    """Split a record into runs of words with the same speaker."""
    segments = []
    for word in record.get("word_timestamps") or []:
        speaker = word.get("speaker") or record.get("speaker") or "?"
        if segments and segments[-1]["speaker"] == speaker:
            segments[-1]["words"].append(word["text"])
            segments[-1]["end"] = word["end"]
        else:
            segments.append({"speaker": speaker, "start": word["start"], "end": word["end"], "words": [word["text"]]})
    return segments


records = all_records(stream)
if not records:
    raise SystemExit("No transcript yet: run Step 5 for a few minutes first.")

# 1. One raw record, as the API returns it (first 3 words shown)
sample = next((r for r in records if len({w.get("speaker") for w in r.get("word_timestamps") or []}) > 1), records[0])
print(json.dumps({**sample, "word_timestamps": sample["word_timestamps"][:3] + ["..."]}, ensure_ascii=False, indent=2))

# 2. Records split by speaker
print(f"\n{len(records)} records")
talk = defaultdict(float)
for number, record in enumerate(records, 1):
    segments = speaker_segments(record)
    mixed = " (mixed: " + ", ".join(s["speaker"] for s in segments) + ")" if len(segments) > 1 else ""
    print(f"\nRecord {number}, speaker {record.get('speaker')}{mixed}")
    for s in segments:
        talk[s["speaker"]] += s["end"] - s["start"]
        clock = datetime.fromtimestamp(s["start"]).strftime("%H:%M:%S")
        print(f"  [{clock}] Speaker {s['speaker']}: {' '.join(s['words'])}")

# 3. Talk time per speaker
print("\nTalk time")
for speaker, seconds in sorted(talk.items()):
    print(f"  Speaker {speaker}: {seconds:6.1f} s")

---
## ⚡ Step 7: Switch to the `pro` Model

To change the model or speaker labels, stop transcription and start it again. Starting with different options while it runs returns an error. The `pro` model formats its text, with capitals and punctuation.

In [ ]:
stream.stop_transcript()
switched_at = time.time()
stream.start_transcript(language_code="en", model_name="pro", speaker_labels=True, max_speakers=4)
show_live_transcript(stream, minutes=3, since=switched_at)

---
## ⏹️ Step 8: Stop the Stream

Stopping also stops transcription and billing. Always stop a stream you no longer need.

In [ ]:
stream.stop()
print("Stream stopped")

---
## ✅ What You Built

- A live transcript that says who is speaking, a few seconds behind the stream.
- Word-level speaker data you can split, count and search.
- The same pipeline on the `pro` model for higher accuracy.

The transcript stays available after the stream stops: call `stream.get_transcript()` any time. Next, index it with `stream.index_spoken_words()` to search what was said.